# 01 — Cleaning the Lending Club dump

This notebook is step one of a credit-risk project I'm building while working through Andrew Ng's ML Specialization. The goal here is narrow: take the raw Lending Club CSV (2007–2018, ~2.26M rows, 145 columns) and produce a smaller, typed, sampled file that notebooks 02 (EDA) and 03 (feature engineering / modeling prep) can use.

I'm deliberately **not** doing feature engineering here. Anything that creates a new feature from existing ones — ratios, encodings, scaling, imputation — lives in `03_feature_engineering.ipynb` so it happens *after* the train/test split. That's the "fit preprocessing on train only" rule from the course, and it's easier to respect if preprocessing lives in one place rather than being sprinkled across every notebook.

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split

pd.set_option("display.max_columns", 50)

## Loading the raw file

The raw CSV is ~1.5 GB and has mixed-type columns that trip pandas' default `low_memory=True` with cryptic dtype warnings. Explicit `low_memory=False` is what works.

In [ ]:
df = pd.read_csv("../data/raw/accepted_2007_to_2018Q4.csv", low_memory=False)
df.shape

In [ ]:
df.head(3)

## What's actually in `loan_status`?

Before deciding what the target is, I want to see every status value and roughly how common it is.

In [ ]:
df["loan_status"].value_counts()

## Which columns to keep, and which to drop as leakage

Lending Club gives you 145 columns, but most fall into one of three unhelpful piles:

1. **Identifiers** (`id`, `member_id`, `url`) — no predictive value, just row keys.
2. **Post-origination columns** — things like `total_pymnt`, `recoveries`, `last_pymnt_amnt`, `collection_recovery_fee`. These only exist *after* the loan has been serviced for some time, so they trivially predict the outcome. The lecture on data leakage in Course 2 is specifically about this situation: if the feature wouldn't have been available at prediction time (loan origination), it can't be in the model.
3. **Columns with mostly missing values** (>40% missing in the full set) — things like `desc` and the various `*_joint` columns. Not worth the trouble on this first pass.

What's left is 32 columns I believe could plausibly be known on the day a loan is issued. I'm being deliberately conservative: if I'm unsure whether a column is pre- or post-origination, I'm dropping it.

In [ ]:
keep_cols = [
    "loan_amnt", "term", "int_rate", "installment", "grade", "sub_grade",
    "emp_length", "home_ownership", "annual_inc", "verification_status",
    "purpose", "addr_state", "dti",
    "delinq_2yrs", "earliest_cr_line", "fico_range_low", "fico_range_high",
    "inq_last_6mths", "mths_since_last_delinq", "mths_since_last_record",
    "open_acc", "pub_rec", "revol_bal", "revol_util", "total_acc",
    "collections_12_mths_ex_med", "acc_now_delinq", "tot_coll_amt",
    "tot_cur_bal", "total_rev_hi_lim",
    "loan_status", "issue_d",
]

keep_cols = [c for c in keep_cols if c in df.columns]
df_clean = df[keep_cols].copy()
df_clean.shape

## Defining the target — what counts as a "default"?

`loan_status` has about ten possible values. I need to collapse them to a binary target: did this loan default or not? Two sub-decisions:

1. **Which statuses count as the positive class.** I'm grouping `Charged Off`, `Default`, `Late (31–120 days)`, `Late (16–30 days)` and the two `Does not meet the credit policy...Charged Off` variants as positive (1 = defaulted). Including the "late" statuses is a judgment call — those borrowers aren't technically defaults yet, they're currently in trouble. For a credit team's operational decision ("is this loan showing distress?") they belong on the same side. I'll check whether this biases probability calibration in notebook 04.
2. **Dropping `Current` loans.** Those are still being paid; we don't know their final outcome. I don't want to make up a label for them, so they come out of the dataset entirely.

In [ ]:
default_statuses = [
    "Charged Off",
    "Default",
    "Late (31-120 days)",
    "Late (16-30 days)",
    "Does not meet the credit policy. Status:Charged Off",
]

completed_statuses = default_statuses + [
    "Fully Paid",
    "Does not meet the credit policy. Status:Fully Paid",
]

df_clean = df_clean[df_clean["loan_status"].isin(completed_statuses)].copy()
df_clean["target"] = df_clean["loan_status"].isin(default_statuses).astype(int)

print(f"rows after filtering: {len(df_clean):,}")
print(f"default rate:         {df_clean['target'].mean()*100:.2f}%")

## Sampling down to 250k rows

The full completed-loan set is around 1.3M rows. I can fit that in memory, but iterating on EDA and modeling gets slow. I'm stratified-sampling 250k rows to keep the class ratio stable while shrinking everything else.

What I lose: the long tail of rare states and loan purposes gets thinner, so a couple of charts in notebook 02 will be noisier for low-volume categories. Worth it for interactivity.

In [ ]:
SAMPLE = 250_000

if len(df_clean) > SAMPLE:
    df_clean, _ = train_test_split(
        df_clean,
        train_size=SAMPLE,
        stratify=df_clean["target"],
        random_state=42,
    )

print(f"after sampling: {df_clean.shape}")
print(f"default rate:   {df_clean['target'].mean()*100:.2f}%")

## Converting a few columns to the right types

Three columns come in as strings but are logically numeric (`term`, `int_rate`, `revol_util`), and `earliest_cr_line` + `issue_d` come in as `MMM-YYYY` strings that I need as real dates — `issue_d` for the time-aware split in notebook 03, and `earliest_cr_line` so 03 can compute credit history as `(issue_d − earliest_cr_line)` per row (an as-of calculation, not a hard-coded reference date).

In [ ]:
df_clean["term"] = df_clean["term"].str.extract(r"(\d+)").astype(float)

for col in ["int_rate", "revol_util"]:
    if df_clean[col].dtype == "object":
        df_clean[col] = df_clean[col].str.replace("%", "", regex=False).astype(float)

# Lending Club uses a 4-digit year, so the format is %b-%Y, not %b-%y.
# Passing %b-%y makes pandas silently misparse "Aug-2003" into the 20YY range.
df_clean["earliest_cr_line_dt"] = pd.to_datetime(
    df_clean["earliest_cr_line"], format="%b-%Y", errors="coerce"
)
df_clean["issue_d_dt"] = pd.to_datetime(
    df_clean["issue_d"], format="%b-%Y", errors="coerce"
)

n_bad = df_clean["earliest_cr_line_dt"].isna().sum()
print(f"earliest_cr_line unparseable: {n_bad:,} of {len(df_clean):,}")

## How bad is the missingness?

Just the summary — I'm not imputing in this notebook (next section explains why).

In [ ]:
missing = df_clean.isna().sum()
missing = missing[missing > 0].sort_values(ascending=False)
missing_pct = (missing / len(df_clean) * 100).round(2)
pd.DataFrame({"missing": missing, "pct": missing_pct})

## Why I'm not imputing in this notebook

Earlier I had `df_clean.fillna(df_clean.median())` sitting right here and it was wrong. Any preprocessing step that fits parameters from the data — a median, a mean, a scaler, a target encoder — has to be fit on the training set *after* the train/test split, or information from the test set leaks into training through those parameters. The specialization makes this explicit in the lecture on applying ML; the practical consequence is that validation metrics look better than production performance, which is exactly the gap you don't want to discover after shipping.

So notebook 01 saves the data **with** its NaN values intact. Notebook 03 does the time-aware split by `issue_d` first, then fits the imputer (plus any scaler / encoder) on the train slice only and applies it to both slices.

Notebook 02's EDA tolerates NaN — `seaborn.histplot` and `pandas.groupby` ignore missing rows by default. I'll flag per-chart where the missing-row count matters.

## What the cleaned dataset looks like

In [ ]:
print(f"shape:        {df_clean.shape}")
print(f"default rate: {df_clean['target'].mean()*100:.2f}%")
print(f"date range:   {df_clean['issue_d_dt'].min():%Y-%m} to {df_clean['issue_d_dt'].max():%Y-%m}")

In [ ]:
df_clean.describe()

## Save for the next notebook

In [ ]:
out_path = "../data/processed/loan_cleaned.csv"
df_clean.to_csv(out_path, index=False)
print(f"wrote {len(df_clean):,} rows to {out_path}")

## One thing that surprised me, one thing I'd do differently

**Surprised me:** how much of this file is leakage. Roughly 40 of the 145 raw columns are only populated after the loan has been serviced — payment histories, recovery amounts, hardship flags. A model that accidentally keeps even one of them (say, `last_pymnt_amnt`) will look amazing in validation and die in production. The leakage lecture made sense in the abstract; this was the first time I had to apply the heuristic column by column on something that wasn't already cleaned for me.

**Would do differently:** stop treating "cleaning" and "feature engineering" as one continuous flow. The first version of this notebook had both mixed together, with imputation and derived features all in one pass, and that's exactly how the median-leakage bug happened — once imputation and FE are entangled, it feels natural to do them both *before* the split. Keeping them in separate notebooks, with the split in between, is a cheap way to make that mistake harder to make.